# Adult Income Classification: Historical Submission

Original individual coursework by Kunze (Charlotte) Lyu. This notebook contains known validation and prior-combination errors; use src/adult_income.py for the reviewed experiment. See docs/REVIEW.md.

## 1. Supervised model training


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder
from sklearn.naive_bayes import GaussianNB, CategoricalNB

train = pd.read_csv("COMP30027_2026_Assignment1_data/Assignment1_data/adult_supervised_train.csv")
con_features = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
cat_features = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']
target = "income"
ALL_FEATURES = con_features + cat_features 
train = train.dropna(subset = ALL_FEATURES) #drop missing values

X_con = train[con_features]
X_cat = train[cat_features]
y = train[target]

# Fit encoder on training categorical features
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value = -1)

def encode_cat(df_cat):
    """Encode categorical features; map unseen/missing to a dedicated OOV slot (index = n_cats)."""
    encoded = encoder.transform(df_cat.fillna('__missing__'))
    for i in range(len(cat_features)) :
        n_cats = len(encoder.categories_[i])
        encoded[:,i] = np.where(encoded[:,i] == -1, n_cats, encoded[:,i])
    return encoded.astype(int)

encoder.fit(X_cat)
X_cat_encoded = encode_cat(X_cat)

gnb = GaussianNB()
cnb = CategoricalNB(alpha = 1.0)
gnb.fit(X_con, y)
cnb.fit(X_cat_encoded, y)

def predict_mixed_nb(x_con,x_cat_encoded,gnb,cnb) :
    """Mixed NB: combine Gaussian (continuous) and Categorical (discrete) log-probs.
    Both models include log prior, so subtract one copy to avoid double-counting."""
    log_prob_con = gnb.predict_log_proba(x_con)
    log_prob_cat = cnb.predict_log_proba(x_cat_encoded)
    log_prob_total = log_prob_con + log_prob_cat
    y_pred = gnb.classes_[np.argmax(log_prob_total, axis=1)]
    return y_pred, log_prob_total

y_pred, log_prob_total = predict_mixed_nb(X_con, X_cat_encoded, gnb, cnb)

# prior probability
print("Prior Probabilities")
for c,p in zip(gnb.classes_, gnb.class_prior_):
    print(f" P({c}) = {p:.4f}")

# Gaussian parameters
print("Gaussian Parameters")
for j, feat in enumerate(con_features):
    for i, c in enumerate(gnb.classes_):
        print(f", {feat} | {c}: mean = {gnb.theta_[i, j]:.2f}, std = {np.sqrt(gnb.var_[i, j]):.2f}")

# R value
print("R value")
c1 = list(gnb.classes_).index('>50K')
c0 = list(gnb.classes_).index('<=50K')
ratios = []
for j, feat in enumerate(cat_features):
    lp = cnb.feature_log_prob_[j]
    for k, val in enumerate(encoder.categories_[j]):
        R = np.exp(lp[c1, k] - lp[c0, k])
        ratios.append((feat, val, R))

rdf =pd.DataFrame(ratios, columns = ['feature', 'value', 'R'])
print("Top 5 for >50k:"); print(rdf.nlargest(5, 'R')[['feature','value', 'R']].to_string(index=False))
print("Top 5 for <=50k:"); print(rdf.nsmallest(5, 'R')[['feature','value', 'R']].to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# Prior Probability Plot
plt.figure()
plt.bar(gnb.classes_, gnb.class_prior_, color = sns.color_palette()[4])
plt.title('Class Prior Probabilities')
plt.ylabel('Probabilities')
plt.savefig('q1_prior_prob.png'); plt.show()

# Continuous feature means plot
other_feats = [f for f in con_features if f != 'capital-gain'] # exclude extreme feature
plt.figure()
x = np.arange(len(other_feats))
plt.bar(x - 0.2, [gnb.theta_[0, con_features.index(f)] for f in other_feats], 0.4, label = gnb.classes_[0])
plt.bar(x + 0.2, [gnb.theta_[1, con_features.index(f)] for f in other_feats], 0.4, label = gnb.classes_[1])
plt.xticks(x, other_feats, rotation = 15)
plt.title('Feature Means by Class')
plt.legend()
plt.savefig('q1_means.png');plt.show()

# Categorical feature R values plot
plt.figure()
top10 = pd.concat([rdf.nlargest(5, 'R'), rdf.nsmallest(5, 'R')])
top10['label'] = top10['feature'] + ': ' + top10['value']
plt.barh(top10['label'], top10['R'], color = sns.color_palette()[4])
plt.axvline(1, color = 'red', linestyle = '--')
plt.title('Top R values')
plt.savefig('q1_rvalues.png');plt.show()

## 2. Supervised model evaluation

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
test = pd.read_csv("COMP30027_2026_Assignment1_data/Assignment1_data/adult_test.csv")

# Fill missing continuous with training median; keep categoricals as it is for OOV detection
X_con_test = test[con_features].fillna(test[con_features].median())
X_cat_test = test[cat_features]
y_test = test[target]

n_categories = [len(c) + 1 for c in encoder.categories_]
cnb = CategoricalNB(alpha = 1.0, min_categories = n_categories)
cnb.fit(X_cat_encoded, y)

# Count OOV
known_cats = [set(encoder.categories_[i]) for i in range(len(cat_features))]
oov_mask = []
for j in range(len(X_cat_test)) :
    has_oov = any(
        X_cat_test.iloc[j, i] not in known_cats[i]
        for i in range(len(cat_features))
    )
    oov_mask.append(has_oov)

print(f"OOV instances: {sum(oov_mask)} / {len(y_test)}")

# Encode test set
X_cat_test_encoded = encode_cat(X_cat_test)

y_pred_test, log_prob_test = predict_mixed_nb(X_con_test, X_cat_test_encoded, gnb, cnb)

# Classification report adn confusion matrix
print("Classification Report:")
print(classification_report(y_test, y_pred_test))
print("Confusion Matrix (rows = actual, columns = predicted):")
print(pd.DataFrame(
    confusion_matrix(y_test, y_pred_test, labels = gnb.classes_),index = gnb.classes_, columns = gnb.classes_
))

# confidence examples using log_R = log(P(y=1|x) / P(y=0|x)) ( R values are too extreme)
log_R = log_prob_test[:, c1] - log_prob_test[:, c0]
test_reset = test.reset_index(drop=True)
key_cols = ['age', 'education', 'occupation', 'marital-status', 'hours-per-week', 'capital-gain']

def print_examples(label, indices):
    print(f"\n{label}:")
    for i in indices:
        row = test_reset.iloc[i]
        print(f" log_R = {log_R[i]:.4f} | true = {y_test.iloc[i]} | pred = {y_pred_test[i]}")
        for col in key_cols:
            print(f" {col}: {row[col]}")

print("\nConfidence Examples:")
print_examples("High confidence >50K", np.argsort(-log_R)[:3])
print_examples("High confidence <= 50K", np.argsort(log_R)[:3])
print_examples("Near boundary (R=1)", np.argsort(np.abs(log_R - 1))[:3])

In [ ]:
def draw_CM(y_true, y_pred, labels, title="Confusion Matrix", cmap = "Blues"):
    """Draw confusion matrix heatmap."""
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    df_cm = pd.DataFrame(cm, index = labels, columns = labels)
    plt.figure(figsize = (7, 4))
    sns.heatmap(df_cm, annot=True, fmt="d", cmap=cmap)
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()
    plt.savefig(title + '.png')

    return cm

draw_CM(y_test, y_pred_test, gnb.classes_, "Supervised Model Confusion Matrix", "YlGnBu")

# Distribution of log_R (outliers removed)
low, high = np.percentile(log_R, 1), np.percentile(log_R, 99)
plt.figure()
plt.hist(log_R, bins = 50, color = 'steelblue', edgecolor = "white", range = (low, high))
plt.axvline(0, color = 'red', linestyle = '--', label = 'boundary')
plt.title('log_R Distribution')
plt.xlabel('log(R)')
plt.legend()
plt.savefig('q2_logr');plt.show()

## 3. Extending the model with semi-supervised training

In [ ]:
# Option 1 Label Propagation
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings("ignore")

unlabelled = pd.read_csv("COMP30027_2026_Assignment1_data/Assignment1_data/adult_unlabelled.csv")
unlabelled = unlabelled.dropna(subset = ALL_FEATURES)

# Split training set into main and validation set for threshold tuning
train_main, val = train_test_split(train, test_size= 0.2, random_state=42, stratify=train[target])

X_con_main = train_main[con_features].values
X_cat_main = encode_cat(train_main[cat_features])
y_main = train_main[target].values

X_con_val = val[con_features].values
X_cat_val = encode_cat(val[cat_features])
y_val = val[target].values

X_con_unlab = unlabelled[con_features].values
X_cat_unlab = encode_cat(unlabelled[cat_features])

# Use Q1 model to assign pseudo-labels; confidence = |log_R|
_, log_prob_unlab = predict_mixed_nb(X_con_unlab, X_cat_unlab, gnb, cnb)
pseudo_labels = gnb.classes_[np.argmax(log_prob_unlab, axis=1)]
log_R_unlab = log_prob_unlab[:, c1] - log_prob_unlab[:, c0]
confidence = np.abs(log_R_unlab)

# Search best confidence threshold on validation F1
print("Threshold search on validation set:")
best_thresh, best_f1 = 0, 0
thresholds = [0, 1, 2, 3, 5, 10, 12, 15, 20]
f1_scores = []
for thresh in thresholds :
    sel = confidence >= thresh
    X_con_comb = np.vstack([X_con_main, X_con_unlab[sel]])
    X_cat_comb = np.vstack([X_cat_main, X_cat_unlab[sel]])
    y_comb = np.concatenate([y_main, pseudo_labels[sel]])

    gnb_tmp = GaussianNB()
    cnb_tmp = CategoricalNB(alpha = 1.0, min_categories=n_categories)
    gnb_tmp.fit(X_con_comb, y_comb)
    cnb_tmp.fit(X_cat_comb, y_comb)

    y_vals_pred, _ = predict_mixed_nb(X_con_val, X_cat_val, gnb_tmp, cnb_tmp)
    f1 = f1_score(y_val, y_vals_pred, pos_label= '>50K')
    f1_scores.append(f1)
    print(f" thresh = {thresh}, selected = {sel.sum()}, val_f1 = {f1:.4f}")
    if f1 > best_f1:
        best_f1, best_thresh = f1, thresh

print(f"\nBest threshold: {best_thresh}, val F1 = {best_f1:.4f}")

# Train final Q3 model on full training set + confident pseudo-labels
X_cat_train_all = encode_cat(train[cat_features])
sel = confidence >= best_thresh

X_con_q3 = np.vstack([train[con_features].values, X_con_unlab[sel]])
X_cat_q3 = np.vstack([X_cat_train_all, X_cat_unlab[sel]])
y_q3 = np.concatenate([train[target].values, pseudo_labels[sel]])

gnb_q3 = GaussianNB()
cnb_q3 = CategoricalNB(alpha = 1.0, min_categories=n_categories)
gnb_q3.fit(X_con_q3, y_q3)
cnb_q3.fit(X_cat_q3, y_q3)

print(f"\nQ3 model trained on {len(y_q3)} instances ({sel.sum()} pseudo-labeled)")

In [ ]:
# Plot validation F1 score vs. confidence threshold
plt.figure()
plt.plot(thresholds, f1_scores, marker = 'o', linestyle = '--')
plt.axvline(best_thresh, color = 'red', linestyle = '--', label = f'best thresh = {best_thresh}')
plt.title('Validation F1 Score vs. Threshold')
plt.xlabel('Threshold')
plt.ylabel('F1 Score')
plt.legend()
plt.savefig('q3_threshold.png');plt.show()

plt.figure()
pseudo_counts = pd.Series(pseudo_labels[confidence >= best_thresh]).value_counts()
plt.bar(pseudo_counts.index, pseudo_counts.values, color = sns.color_palette()[4])
plt.title('Pseudo Label Distribution')
plt.savefig('q3_pseudo.png');plt.show()

## 4. Supervised model evaluation

In [ ]:
y_pred_q3, log_prob_q3 = predict_mixed_nb(X_con_test, X_cat_test_encoded, gnb_q3, cnb_q3)

# Performance comparison
print("Q1 Supervised:")
print(classification_report(y_test, y_pred_test))

print("Q3 Semi-supervised: (Label Propagation, threshold = 12):")
print(classification_report(y_test, y_pred_q3))

print("Confusion Matrix Q3:")
print(pd.DataFrame(
    confusion_matrix(y_test, y_pred_q3, labels = gnb_q3.classes_),index = gnb_q3.classes_, columns = gnb_q3.classes_
))

# Posterior ratio distribution shift
log_R_q1 = log_prob_test[:, c1] - log_prob_test[:, c0]
log_R_q3 = log_prob_q3[:, c1] - log_prob_q3[:, c0]
print(f"\nQ1 log_R: mean= {log_R_q1.mean():.3f}, std = {log_R_q1.std():.3f}")
print(f"Q3 log_R: mean= {log_R_q3.mean():.3f}, std = {log_R_q3.std():.3f}")

# Gaussian parameter changes
print("\nGaussian parameter changes (Q1 -> Q3):")
for j, feat in enumerate(con_features):
    for i, c in enumerate(gnb.classes_):
        m1 = gnb.theta_[i, j]
        m3 = gnb_q3.theta_[i, j]
        print(f" {feat} | {c}: {m1:.2f} -> {m3:.2f} (delta = {m3 - m1:.2f})")

# Most predictive categorical features comparison
print("\nTop 5 for >50K (Q3):")
ratios_q3 = []
for j, feat in enumerate(cat_features):
    lp = cnb_q3.feature_log_prob_[j]
    for k, val in enumerate(encoder.categories_[j]):
        R = np.exp(lp[c1, k] - lp[c0, k])
        ratios_q3.append((feat, val, R))
rdf_q3 = pd.DataFrame(ratios_q3, columns = ['feature', 'value', 'R'])
print(rdf_q3.nlargest(5, 'R')[['feature','value', 'R']].to_string(index=False))
print("\nTop 5 for <=50K (Q3):")
print(rdf_q3.nsmallest(5, 'R')[['feature','value', 'R']].to_string(index=False))

In [ ]:
# log_R distribution comparison (outliers removed)
low1, high1 = np.percentile(log_R_q1, 1), np.percentile(log_R_q1, 99)
low3, high3 = np.percentile(log_R_q3, 1), np.percentile(log_R_q3, 99)
loww, highh = min(low1, low3), max(high1, high3)

plt.figure()
plt.hist(log_R_q1, bins = 50, alpha = 0.6, label = 'Q1', color = 'steelblue', range = (loww, highh))
plt.hist(log_R_q3, bins = 50, alpha = 0.6, label = 'Q3', color = 'orange', range = (loww, highh))
plt.axvline(0, color = 'red', linestyle = '--')
plt.title('log_R Distribution (Q1 vs. Q3)')
plt.xlabel('log_R')
plt.legend()
plt.savefig('q4_logr.png');plt.show()

# Gaussian mean delta per feature per class
plt.figure()
x = np.arange(len(con_features))
deltas_le = [gnb_q3.theta_[0, j] - gnb.theta_[0, j] for j in range(len(con_features)) ]
deltas_gt = [gnb_q3.theta_[1, j] - gnb.theta_[1, j] for j in range(len(con_features)) ]
plt.bar(x - 0.2, deltas_le, 0.4, label = '<=50K')
plt.bar(x + 0.2, deltas_gt, 0.4, label = '>50K')
plt.xticks(x, con_features, rotation = 15)
plt.yscale('symlog')
plt.axhline(0, color = 'black', linewidth = 0.8)
plt.title('Gaussian Mean Change (Q1-Q3)')
plt.legend()
plt.savefig('q4_means.png');plt.show()

# Confusion matrix for Q3 model
draw_CM(y_test, y_pred_q3, labels = gnb_q3.classes_, title = "Q3 Semi-supervised Model Confusion Matrix", cmap = "YlGnBu")